# 02 Compare Layers

This notebook inspects the current corpus scaffold and begins comparing how the same packet is framed across layers.

The goal is not sophisticated modeling. The goal is to make packet structure, layer coverage, and a few early framing differences visible.


In [1]:
from pathlib import Path
from collections import Counter
import re

import pandas as pd

repo_root = Path('..').resolve()
documents_path = repo_root / 'data' / 'processed' / 'documents.csv'
links_path = repo_root / 'data' / 'processed' / 'links.csv'

documents_df = pd.read_csv(documents_path)
links_df = pd.read_csv(links_path)
documents_df['has_text'] = documents_df['text_clean'].fillna('').str.len() > 0


## Coverage Summary

This shows how much real working text has been loaded into each packet and layer.


In [2]:
coverage = (
    documents_df.groupby(['packet_id', 'layer', 'has_text'])
    .size()
    .reset_index(name='count')
)
display(coverage)


,packet_id,layer,has_text,count
0,CN-02,news,True,3
1,CN-02,official,True,1
2,CN-02,social,True,1
3,CN-04,news,True,3
4,CN-04,official,True,1
5,CN-04,social,True,3
6,US-01,news,True,3
7,US-01,official,True,1
8,US-02,news,True,3
9,US-02,official,True,1


In [3]:
packet_matrix = (
    documents_df[documents_df['has_text']]
    .groupby(['packet_id', 'layer'])
    .size()
    .unstack(fill_value=0)
)
display(packet_matrix)


layer,news,official,social
packet_id,,,
CN-02,3,1,1
CN-04,3,1,3
US-01,3,1,0
US-02,3,1,2
US-03,3,1,3


## Link Scaffold

Cross-layer links connect news and social documents to their corresponding official anchors. The current scaffold is inferred from packet membership; individual link verification is noted in the `notes` field.


In [4]:
display(links_df.groupby('relation_type').size().reset_index(name='count'))
display(links_df.head(12))


,relation_type,count
0,reports_on,15
1,reposts_or_discusses,9


,link_id,from_doc_id,to_doc_id,relation_type,evidence,confidence,notes
0,us-01__news__1,us-01__news__news_1,us-01__official__anchor,reports_on,official_news_release_on_same_guidance,0.95,Official NewsNet release tied directly to the ...
1,us-01__news__2,us-01__news__news_2,us-01__official__anchor,reports_on,workflow_explainer_on_same_guidance,0.90,Official explainer for the same guidance and r...
2,us-01__news__3,us-01__news__news_3,us-01__official__anchor,reports_on,packet_membership,0.30,Link inferred from shared packet membership.
3,us-02__news__1,us-02__news__news_1,us-02__official__anchor,reports_on,packet_membership,0.30,Link inferred from shared packet membership.
4,us-02__news__2,us-02__news__news_2,us-02__official__anchor,reports_on,packet_membership,0.30,Link inferred from shared packet membership.
5,us-02__news__3,us-02__news__news_3,us-02__official__anchor,reports_on,packet_membership,0.30,Link inferred from shared packet membership.
6,us-02__social__1,us-02__social__social_1,us-02__official__anchor,reposts_or_discusses,packet_membership,0.20,Packet-level link; individual verification pen...
7,us-02__social__2,us-02__social__social_2,us-02__official__anchor,reposts_or_discusses,packet_membership,0.20,Packet-level link; individual verification pen...
8,us-03__news__1,us-03__news__news_1,us-03__official__anchor,reports_on,official_news_release_on_same_report,0.98,Official NewsNet release announcing the same P...
9,us-03__news__2,us-03__news__news_2,us-03__official__anchor,reports_on,general_news_on_same_report,0.85,General news coverage summarizing the same Cop...


## Packet Comparison Helpers

These helper functions extract rough top terms for a given packet and layer. They are only for first-pass inspection.


In [5]:
STOPWORDS = {
    'the','and','that','this','with','for','are','its','can','not','into','from','their','they','which','than',
    'have','will','through','more','only','such','because','where','when','used','using','also','while','under',
    'it','is','of','to','a','in','on','or','as','by','be','an','at','how','what','who','does','do','if','but'
}

def top_terms(packet_id, layer, top_n=12):
    subset = documents_df[(documents_df['packet_id'] == packet_id) & (documents_df['layer'] == layer) & (documents_df['has_text'])]
    tokens = []
    for text in subset['text_clean'].fillna(''):
        tokens.extend(re.findall(r'[A-Za-z\u4e00-\u9fff]+', text.lower()))
    tokens = [token for token in tokens if token not in STOPWORDS and len(token) >= 2]
    return Counter(tokens).most_common(top_n)

def packet_snapshot(packet_id):
    packet_rows = documents_df[(documents_df['packet_id'] == packet_id) & (documents_df['has_text'])]
    return packet_rows[['doc_id', 'layer', 'title']]


## U.S. Packet: US-03

US-03 covers the Copyright Office's Part 2 copyrightability report. It includes one official anchor, three news articles, and three social posts, providing full layer coverage.


In [6]:
display(packet_snapshot('US-03'))
print('official', top_terms('US-03', 'official'))
print('news', top_terms('US-03', 'news'))


,doc_id,layer,title
10,us-03__official__anchor,official,"Copyright and Artificial Intelligence, Part 2:..."
11,us-03__news__news_1,news,Copyright Office Releases Part 2 of Artificial...
12,us-03__news__news_2,news,AI-assisted works can get copyright with enoug...
13,us-03__news__news_3,news,Purely AI-generated art can't get copyright pr...
14,us-03__social__social_1,social,Nicole Hennig Bluesky summary of the report's ...
15,us-03__social__social_2,social,Polygon Bluesky summary of human authorship re...
16,us-03__social__social_3,social,Future of Music Coalition Bluesky post on disc...


official [('copyright', 461), ('ai', 367), ('comments', 311), ('office', 236), ('initial', 209), ('human', 149), ('generated', 139), ('work', 131), ('see', 102), ('response', 99), ('notice', 96), ('output', 95)]
news [('ai', 45), ('copyright', 35), ('office', 26), ('human', 19), ('report', 14), ('work', 14), ('generated', 14), ('output', 11), ('works', 11), ('protection', 8), ('protected', 7), ('creative', 7)]


## Preliminary Observations

- The official layer foregrounds terms related to legal criteria: `human`, `control`, `copyrightability`.
- The news layer shifts toward outcome-oriented and accessible language: `protected`, `author`, and public-facing explanations.


## China Packet: CN-02

CN-02 covers the Beijing Internet Court's AI-generated image copyright case. It includes one official anchor, news coverage from multiple outlets, and a tech media social post.


In [7]:
display(packet_snapshot('CN-02'))
print('official', top_terms('CN-02', 'official'))
print('news', top_terms('CN-02', 'news'))
print('social', top_terms('CN-02', 'social'))


,doc_id,layer,title
17,cn-02__official__anchor,official,北京互联网法院 AI 文生图著作权案一审判决
18,cn-02__news__news_1,news,AI 文生图有版权吗？北京互联网法院一审判了
19,cn-02__news__news_2,news,AI绘画有版权吗？北京互联网法院判了
20,cn-02__news__news_3,news,AI文生图著作权案落槌，判决鼓励新技术发展
21,cn-02__social__social_1,social,IT之家报道北京互联网法院首例AI文生图著作权案庭审


official [('quality', 8), ('safetensors', 7), ('extra', 7), ('https', 6), ('所示', 6), ('stable', 5), ('asiafacemix', 5), ('模糊', 5), ('突变', 5), ('com', 4), ('diffusion', 4), ('hanfugirl', 4)]
news [('作品', 4), ('属性和使用者的', 4), ('创作者', 4), ('身份', 4), ('享有涉案图片的著作权', 3), ('近期', 2), ('北京互联网法院审结了李某与刘某侵害作品署名权和信息网络传播权纠纷一案', 2), ('明确了利用人工智能生成图片的', 2), ('法院经审理认为', 2), ('受到著作权法的保护', 2), ('原告是涉案图片的作者', 2), ('北京互联网法院作出一审判决', 2)]
social [('ai文生图', 2), ('it之家', 1), ('日消息', 1), ('据北京互联网法院官方账号发文表示', 1), ('北京互联网法院近日依法公开开庭审理了一起', 1), ('著作权案', 1), ('该案为我国首例涉', 1), ('it之家从官方账号处得知', 1), ('该案庭审过程由中央广播电视总台新闻中心进行全媒体直播', 1), ('清华大学法学院知识产权法研究中心主任崔国斌教授', 1), ('计算机图像算法工程师黄影元作为节目嘉宾', 1), ('对ai生成内容涉及的技术与法律问题进行了讨论', 1)]


## Preliminary Observations

- The official layer foregrounds terms related to the court's analytical framework: `intellectual input`, `selection`, `personalized expression`.
- The news layer compresses the reasoning into a result-oriented frame: `copyright`, `author`, `decision`.
- The social layer provides a more abbreviated public-facing summary of the case outcome.


## China Adjacent Packet: CN-04

CN-04 covers the synthetic content labeling measures. Unlike the authorship-focused packets, this one illustrates an operational regulatory propagation pattern: from official notice to media explanation to social redistribution.


In [8]:
display(packet_snapshot('CN-04'))
print('official', top_terms('CN-04', 'official'))
print('news', top_terms('CN-04', 'news'))
print('social', top_terms('CN-04', 'social'))


,doc_id,layer,title
22,cn-04__official__anchor,official,关于印发《人工智能生成合成内容标识办法》的通知
23,cn-04__news__news_1,news,《人工智能生成合成内容标识办法》发布 将于9月1日起施行
24,cn-04__news__news_2,news,国家互联网信息办公室等四部门联合发布《人工智能生成合成内容标识办法》
25,cn-04__news__news_3,news,AI生成合成内容标识办法发布 有何重要意义？
26,cn-04__social__social_1,social,中国新闻周刊微博发布标识办法要点
27,cn-04__social__social_2,social,新浪科技 Weibo post on the labeling rule
28,cn-04__social__social_3,social,法治网微博转发标识办法施行通知


official [('互联网信息服务深度合成管理规定', 4), ('采取适当方式在发布内容周边添加显著的提示标识', 3), ('互联网信息服务算法推荐管理规定', 2), ('生成式人工智能服务管理暂行办法', 2), ('文件元数据中未核验到隐式标识', 2), ('行政法规', 2), ('第一条', 1), ('为了促进人工智能健康发展', 1), ('规范人工智能生成合成内容标识', 1), ('保护公民', 1), ('法人和其他组织合法权益', 1), ('维护社会公共利益', 1)]
news [('标识办法', 6), ('办法', 6), ('人工智能生成合成内容标识办法', 4), ('图片', 4), ('音频', 4), ('规范人工智能生成合成内容标识', 3), ('公安部', 3), ('采取适当方式在发布内容周边添加显著的提示标识', 3), ('为了促进人工智能健康发展', 2), ('工业和信息化部', 2), ('国家广播电视总局制定了', 2), ('将于', 2)]
social [('人工智能生成合成内容标识办法', 6), ('图片', 3), ('音频', 3), ('国家互联网信息办公室', 2), ('工业和信息化部', 2), ('公安部', 2), ('日起施行', 2), ('办法', 2), ('视频', 2), ('四部门联合发布', 1), ('国家广播电视总局联合发布', 1), ('将于', 1)]


## Preliminary Observations

- This packet demonstrates a regulation-to-media-to-social propagation pattern, with operational language (labeling, metadata, compliance) remaining prominent across all layers.
- As an adjacent packet, it addresses content governance rather than core authorship doctrine, providing a comparative reference for how regulatory text transforms differently from judicial reasoning.


## Limitations and Future Work

- **Link granularity**: Current cross-layer links are inferred from packet membership. Future iterations should refine these into more specific `news → official` and `social → news` relations with evidentiary support.
- **Corpus scope**: The corpus is intentionally narrow, covering five packets across two jurisdictions. Expansion to additional jurisdictions or regulatory domains (e.g., EU AI Act, education policy) would require corresponding manifest and pipeline extensions.
- **Text coverage**: Some social layer sources rely on reconstructed text where original posts are no longer publicly accessible. These entries are noted in the raw file metadata.
- **Analytical depth**: The top-term extraction in this notebook provides a first-pass overview. More rigorous analysis would benefit from domain-specific stopword lists, TF-IDF weighting, or embedding-based similarity measures.
